#  Customer Support Ticket Pipeline + RAG Assistant
**Modern Data Engineering for AI Systems – Final Project (SDAIA Academy)**

✅***This project was created as part of the "Modern Data Engineering for AI Systems program" at SDAIA Academy.***

Pipeline: `CSV → Spark → Quality Checks → Quality Gate → (FAIL: Quarantine | PASS: Delta Lake clean) → RAG (Embeddings + ChromaDB + LLM)`


## 0) Install & setup

In [29]:
!pip install -q pyspark==3.5.3 delta-spark==3.2.1 sentence-transformers chromadb anthropic

In [30]:
import os, shutil, random
from datetime import datetime, timedelta
import pandas as pd

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.appName("SupportTicketsPipeline")
           .master("local[*]")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark is ready")

Spark is ready


In [31]:
# ---------- Config ----------
BASE = "/content"
DATA_DIR = f"{BASE}/data"
RAW_PATH = f"{BASE}/lakehouse/raw_tickets"
CLEAN_PATH = f"{BASE}/lakehouse/clean_tickets"
QUARANTINE_PATH = f"{BASE}/lakehouse/quarantine_tickets"

GATE_THRESHOLD = 0.05          # PASS if failing rows <= 5%
VALID_PRIORITIES = ["low", "medium", "high"]
MIN_RESOLUTION_LEN = 10

# Start fresh every run (so the demo is repeatable)
shutil.rmtree(f"{BASE}/lakehouse", ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

## 1) Get a dataset (synthetic support tickets)
Two batches:
- `tickets_good.csv` → almost clean (1% errors) → **PASS**
- `tickets_bad.csv` → many errors (nulls, duplicates, invalid priority, short resolution) → **FAIL → Quarantine**

In [32]:
TEMPLATES = {
    "billing": [
        ("I was charged twice for my order", "Duplicate charge refunded within 3-5 business days after verifying the payment logs."),
        ("Why is my invoice amount higher than expected?", "Check the invoice breakdown in Billing; taxes and shipping are added at checkout. Contact us if the total still looks wrong."),
        ("How can I get a refund for my subscription?", "Cancel from Account > Subscription and request a refund within 14 days of the charge."),
    ],
    "login": [
        ("I cannot log in to my account", "Reset your password from the login page using the Forgot Password link, then sign in again."),
        ("I did not receive the verification code", "Wait 2 minutes, check spam, then request a new code. Make sure your phone number is correct in your profile."),
        ("My account is locked after several attempts", "Accounts unlock automatically after 30 minutes. You can also reset your password to unlock it immediately."),
    ],
    "shipping": [
        ("Where is my order? It has not arrived yet", "Track the shipment using the tracking number in your confirmation email. Delays of up to 2 days can happen during peak season."),
        ("Can I change my delivery address?", "Address changes are possible before the order ships. Go to Orders > Edit address or contact support."),
        ("The package arrived damaged", "We are sorry! Send a photo of the package from the Orders page and we will ship a replacement at no cost."),
    ],
    "returns": [
        ("How do I return a product?", "Open Orders > Return item, print the label, and drop the package at any carrier point within 30 days."),
        ("I received the wrong item", "Please upload a photo of the item from the Orders page. We will send the correct item and arrange a free pickup."),
        ("When will I get my money back after a return?", "Refunds are issued 5-7 business days after the returned item is received and inspected."),
    ],
}
PREFIXES = ["", "", "Hi, ", "Hello, ", "Please help: "]

def make_tickets(n, start_id, seed):
    rnd = random.Random(seed)
    rows = []
    for i in range(n):
        cat = rnd.choice(list(TEMPLATES))
        q, r = rnd.choice(TEMPLATES[cat])
        pr = rnd.choice(VALID_PRIORITIES)
        rows.append({
            "ticket_id": f"T{start_id + i:05d}",
            "customer_id": f"C{rnd.randint(1, 60):03d}",
            # messy raw values on purpose (fixed later in the transformation step)
            "category": (cat.upper() + " ") if rnd.random() < 0.2 else cat,
            "question": rnd.choice(PREFIXES) + q,
            "resolution": r,
            "priority": pr.capitalize() if rnd.random() < 0.2 else pr,
            "created_at": (datetime(2026, 9, 1) + timedelta(minutes=rnd.randint(0, 40000))).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

# ---- GOOD batch: 200 rows, only 2 bad rows (1%) ----
good = make_tickets(200, start_id=1, seed=42)
good.loc[5, "question"] = None          # completeness issue
good.loc[17, "priority"] = "urgent"     # validity issue
good.to_csv(f"{DATA_DIR}/tickets_good.csv", index=False)

# ---- BAD batch: lots of problems ----
bad = make_tickets(100, start_id=1001, seed=7)
bad.loc[0:14, "question"] = None              # completeness
bad.loc[15:29, "resolution"] = None           # completeness
bad.loc[30:44, "priority"] = "asap"           # validity
bad.loc[45:54, "resolution"] = "ok"           # business rule (too short)
bad = pd.concat([bad, bad.iloc[60:70]])       # uniqueness (duplicate ticket_id)
bad.to_csv(f"{DATA_DIR}/tickets_bad.csv", index=False)

print("good:", len(good), "rows | bad:", len(bad), "rows")
good.head()

good: 200 rows | bad: 110 rows


,ticket_id,customer_id,category,question,resolution,priority,created_at
0,T00001,C018,billing,I was charged twice for my order,Duplicate charge refunded within 3-5 business ...,high,2026-09-25 19:41:00
1,T00002,C003,BILLING,How can I get a refund for my subscription?,Cancel from Account > Subscription and request...,medium,2026-09-28 09:33:00
2,T00003,C046,billing,Please help: How can I get a refund for my sub...,Cancel from Account > Subscription and request...,low,2026-09-21 10:39:00
3,T00004,C045,shipping,"Hi, Where is my order? It has not arrived yet",Track the shipment using the tracking number i...,Low,2026-09-16 07:39:00
4,T00005,C007,billing,"Hi, I was charged twice for my order",Duplicate charge refunded within 3-5 business ...,medium,2026-09-02 23:27:00


## 2) Ingestion – Load data with Spark → Delta (raw / bronze)

In [33]:
SCHEMA = StructType([StructField(c, StringType(), True) for c in
    ["ticket_id", "customer_id", "category", "question", "resolution", "priority", "created_at"]])

def ingest(csv_path, batch_id):
    df = (spark.read.option("header", True).schema(SCHEMA).csv(csv_path)
          .withColumn("batch_id", F.lit(batch_id))
          .withColumn("ingest_ts", F.current_timestamp()))
    df.write.format("delta").mode("append").save(RAW_PATH)   # keep an untouched copy of the raw data
    print(f"[{batch_id}] ingested {df.count()} rows into raw Delta table")
    return df

## 3) Data Quality checks (4 checks)
| Check | Dimension | Rule |
|---|---|---|
| `fail_completeness` | Completeness | `question` and `resolution` must not be null/empty |
| `fail_uniqueness` | Uniqueness | `ticket_id` must be unique |
| `fail_validity` | Validity | `priority` ∈ {low, medium, high} |
| `fail_business_rule` | Accuracy / Business rule | `resolution` length ≥ 10 characters |

In [34]:
CHECK_COLS = ["fail_completeness", "fail_uniqueness", "fail_validity", "fail_business_rule"]

def run_quality_checks(df):
    w = Window.partitionBy("ticket_id").orderBy("created_at")
    priority_ok = F.coalesce(F.lower(F.trim("priority")).isin(VALID_PRIORITIES), F.lit(False))

    checked = (df
        .withColumn("_rn", F.row_number().over(w))
        .withColumn("fail_completeness", F.when(
            F.col("question").isNull() | (F.trim("question") == "") |
            F.col("resolution").isNull() | (F.trim("resolution") == ""), 1).otherwise(0))
        .withColumn("fail_uniqueness", F.when(F.col("_rn") > 1, 1).otherwise(0))
        .withColumn("fail_validity", F.when(~priority_ok, 1).otherwise(0))
        .withColumn("fail_business_rule", F.when(F.length(F.trim("resolution")) < MIN_RESOLUTION_LEN, 1).otherwise(0))
    )
    checked = (checked
        .withColumn("fail_reasons", F.concat_ws(",",
            F.when(F.col("fail_completeness") == 1, F.lit("completeness")),
            F.when(F.col("fail_uniqueness") == 1, F.lit("uniqueness")),
            F.when(F.col("fail_validity") == 1, F.lit("validity")),
            F.when(F.col("fail_business_rule") == 1, F.lit("business_rule"))))
        .withColumn("is_bad", F.when(F.length("fail_reasons") > 0, 1).otherwise(0)))
    return checked

## 4) Transformation: raw → clean/trusted
(trim text, normalize category/priority, cast timestamp, build `doc_text` for RAG)

In [35]:
def transform(good_df):
    return (good_df
        .withColumn("category", F.lower(F.trim("category")))
        .withColumn("priority", F.lower(F.trim("priority")))
        .withColumn("question", F.trim("question"))
        .withColumn("resolution", F.trim("resolution"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("doc_text", F.concat_ws(" | ",
            F.concat(F.lit("Category: "), F.col("category")),
            F.concat(F.lit("Question: "), F.col("question")),
            F.concat(F.lit("Resolution: "), F.col("resolution"))))
        .select("ticket_id", "customer_id", "category", "priority", "question",
                "resolution", "created_at", "doc_text", "batch_id"))

## 5) Quality Gate: PASS / FAIL
- **PASS** (error rate ≤ 5%): good rows → transform → **Delta clean table**; the few bad rows → Quarantine
- **FAIL** (error rate > 5%): the **whole batch** → **Quarantine**, nothing reaches the clean table

In [36]:
def quality_gate(checked, batch_id, threshold=GATE_THRESHOLD):
    checked = checked.cache()
    total = checked.count()
    rep = checked.agg(*[F.sum(c).alias(c) for c in CHECK_COLS],
                      F.sum("is_bad").alias("bad_rows")).collect()[0].asDict()
    bad = int(rep["bad_rows"] or 0)
    rate = bad / total
    status = "PASS" if rate <= threshold else "FAIL"

    print("=" * 55)
    print(f"Batch: {batch_id} | total rows: {total}")
    for c in CHECK_COLS:
        print(f"  {c:<22} failed rows: {int(rep[c] or 0)}")
    print(f"  error rate: {rate:.1%} (threshold {threshold:.0%})  ->  {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
    print("=" * 55)

    to_quarantine = checked if status == "FAIL" else checked.filter("is_bad = 1")
    if to_quarantine.head(1):
        (to_quarantine.drop("_rn")
            .withColumn("gate_status", F.lit(status))
            .withColumn("quarantined_at", F.current_timestamp())
            .write.format("delta").mode("append").save(QUARANTINE_PATH))
        print(f"-> {'whole batch' if status == 'FAIL' else 'bad rows'} sent to QUARANTINE")

    if status == "PASS":
        clean = transform(checked.filter("is_bad = 0"))
        clean.write.format("delta").mode("append").save(CLEAN_PATH)
        print(f"-> {clean.count()} rows written to CLEAN Delta table")
    return status

def run_pipeline(csv_path, batch_id):
    raw = ingest(csv_path, batch_id)
    checked = run_quality_checks(raw)
    return quality_gate(checked, batch_id)

## 6) Run the pipeline: one PASS case and one FAIL case

In [23]:
# ✅ Passing case
status_good = run_pipeline(f"{DATA_DIR}/tickets_good.csv", "batch_good")

[batch_good] ingested 200 rows into raw Delta table
Batch: batch_good | total rows: 200
  fail_completeness      failed rows: 1
  fail_uniqueness        failed rows: 0
  fail_validity          failed rows: 1
  fail_business_rule     failed rows: 0
  error rate: 1.0% (threshold 5%)  ->  ✅ PASS
-> bad rows sent to QUARANTINE
-> 198 rows written to CLEAN Delta table


In [24]:
# ❌ Failing case
status_bad = run_pipeline(f"{DATA_DIR}/tickets_bad.csv", "batch_bad")

[batch_bad] ingested 110 rows into raw Delta table
Batch: batch_bad | total rows: 110
  fail_completeness      failed rows: 30
  fail_uniqueness        failed rows: 10
  fail_validity          failed rows: 15
  fail_business_rule     failed rows: 10
  error rate: 59.1% (threshold 5%)  ->  ❌ FAIL
-> whole batch sent to QUARANTINE


### Inspect the Delta tables

In [25]:
print("CLEAN table:")
clean_df = spark.read.format("delta").load(CLEAN_PATH)
print("rows:", clean_df.count())
clean_df.select("ticket_id", "category", "priority", "question").show(5, truncate=60)

print("QUARANTINE summary (by batch and reason):")
q_df = spark.read.format("delta").load(QUARANTINE_PATH)
(q_df.groupBy("batch_id", "gate_status", "fail_reasons").count()
     .orderBy("batch_id", F.desc("count")).show(truncate=False))

print("Delta history of the clean table (lineage / versioning):")
DeltaTable.forPath(spark, CLEAN_PATH).history().select("version", "timestamp", "operation").show(truncate=False)

CLEAN table:
rows: 198
+---------+--------+--------+-----------------------------------------------------+
|ticket_id|category|priority|                                             question|
+---------+--------+--------+-----------------------------------------------------+
|   T00134| billing|     low|Hello, Why is my invoice amount higher than expected?|
|   T00192| billing|    high|Hello, Why is my invoice amount higher than expected?|
|   T00068| returns|  medium|              Please help: How do I return a product?|
|   T00150|   login|     low|                        I cannot log in to my account|
|   T00115|   login|  medium|       Hello, I did not receive the verification code|
+---------+--------+--------+-----------------------------------------------------+
only showing top 5 rows

QUARANTINE summary (by batch and reason):
+----------+-----------+-------------+-----+
|batch_id  |gate_status|fail_reasons |count|
+----------+-----------+-------------+-----+
|batch_bad |FAIL   

## 7) AI / RAG output: support assistant
Embeddings (`sentence-transformers`, multilingual) → vector DB (**ChromaDB**) → retrieval → **LLM** answer.
Only **trusted** data from the clean Delta table is indexed.

In [26]:
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

tickets = spark.read.format("delta").load(CLEAN_PATH).toPandas()

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("support_tickets", metadata={"hnsw:space": "cosine"})

embeddings = embed_model.encode(tickets["question"].tolist(), normalize_embeddings=True, show_progress_bar=False).tolist()
collection.upsert(
    ids=tickets["ticket_id"].tolist(),
    embeddings=embeddings,
    documents=tickets["doc_text"].tolist(),
    metadatas=[{"category": c, "resolution": r} for c, r in zip(tickets["category"], tickets["resolution"])],
)
print("Indexed", collection.count(), "tickets in ChromaDB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indexed 198 tickets in ChromaDB


In [27]:
# Optional: LLM key. In Colab: left sidebar 🔑 Secrets -> add ANTHROPIC_API_KEY (enable notebook access).
# Without a key, the assistant falls back to the best retrieved resolution (still a valid RAG retrieval demo).
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
try:
    from google.colab import userdata
    API_KEY = API_KEY or userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

def retrieve(question, k=3):
    q_emb = embed_model.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return [{"doc": d, "meta": m, "distance": dist}
            for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

def generate_answer(question, hits):
    context = "\n".join(f"- {h['doc']}" for h in hits)
    prompt = (
        "You are a customer support assistant. Answer the customer's question using ONLY the context "
        "from previous resolved tickets. If the context is not enough, say so. "
        "Reply in the same language as the customer.\n\n"
        f"Context:\n{context}\n\nCustomer question: {question}\nAnswer:"
    )
    if API_KEY:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=API_KEY)
            msg = client.messages.create(model="claude-sonnet-5-5", max_tokens=300,
                                         messages=[{"role": "user", "content": prompt}])
            return msg.content[0].text
        except Exception as e:
            print("LLM call failed, using fallback:", e)
    return "(fallback, no LLM) " + hits[0]["meta"]["resolution"]

def ask(question, k=3):
    hits = retrieve(question, k)
    print("❓", question)
    print("🔎 Retrieved tickets:")
    for h in hits:
        print(f"   - [{h['meta']['category']}] (distance {h['distance']:.3f}) {h['doc'][:90]}...")
    print("💬 Answer:", generate_answer(question, hits), "\n")

ask("My card was billed two times for the same purchase")
ask("I forgot my password and can't sign in")
ask("وين طلبي؟ تأخر كثير")

❓ My card was billed two times for the same purchase
🔎 Retrieved tickets:
   - [billing] (distance 0.265) Category: billing | Question: I was charged twice for my order | Resolution: Duplicate cha...
   - [billing] (distance 0.265) Category: billing | Question: I was charged twice for my order | Resolution: Duplicate cha...
   - [billing] (distance 0.265) Category: billing | Question: I was charged twice for my order | Resolution: Duplicate cha...
💬 Answer: (fallback, no LLM) Duplicate charge refunded within 3-5 business days after verifying the payment logs. 

❓ I forgot my password and can't sign in
🔎 Retrieved tickets:
   - [login] (distance 0.236) Category: login | Question: I cannot log in to my account | Resolution: Reset your passwor...
   - [login] (distance 0.236) Category: login | Question: I cannot log in to my account | Resolution: Reset your passwor...
   - [login] (distance 0.236) Category: login | Question: I cannot log in to my account | Resolution: Reset your passwor..

## 8) Summary
- Batch `batch_good` → **PASS** → rows in the clean Delta table → indexed in the vector DB → used by the RAG assistant
- Batch `batch_bad` → **FAIL** → entire batch isolated in the Quarantine Delta table with failure reasons

In [28]:
print("batch_good:", status_good, "| batch_bad:", status_bad)
print("raw rows       :", spark.read.format("delta").load(RAW_PATH).count())
print("clean rows     :", spark.read.format("delta").load(CLEAN_PATH).count())
print("quarantine rows:", spark.read.format("delta").load(QUARANTINE_PATH).count())

batch_good: PASS | batch_bad: FAIL
raw rows       : 310
clean rows     : 198
quarantine rows: 112
